## Динамическое программирование и Левенштейн

Динамическое программирование (ДП) — это метод, при котором сложные задачи разбиваются на небольшие подзадачи. Таким образом, достичь сложной цели становится проще. Решения подзадач сохраняются и используются в будущем для решения других задач. Это позволяет избежать повторных вычислений и значительно ускорить процесс.

Приведу простой пример. Представьте, что перед вами огромная мозаика. Чтобы её собрать, нужно постепенно складывать элемент за элементом, пока не получится целая картинка. Проще всего взять несколько объектов мозаики, собрать их и потом соединить. Динамическое программирование можно сравнить с укладкой мозаики.

Словосочетание «динамическое программирование» впервые было использовано в 1940-х годах Ричардом Беллманом для описания процесса нахождения решения задачи, где ответ на одну задачу может быть получен только после решения задачи, «предшествующей» ей. В 1953 году он уточнил это определение до современного.

Слово «программирование» в словосочетании «динамическое программирование» в действительности к «традиционному» программированию (написанию кода) почти никакого отношения не имеет и имеет смысл как в словосочетании «математическое программирование», которое является синонимом слова «оптимизация». Поэтому слово «программа» в данном контексте скорее означает оптимальную последовательность действий для получения решения задачи. ([Материал Вики](https://ru.wikipedia.org/wiki/%D0%94%D0%B8%D0%BD%D0%B0%D0%BC%D0%B8%D1%87%D0%B5%D1%81%D0%BA%D0%BE%D0%B5_%D0%BF%D1%80%D0%BE%D0%B3%D1%80%D0%B0%D0%BC%D0%BC%D0%B8%D1%80%D0%BE%D0%B2%D0%B0%D0%BD%D0%B8%D0%B5))

### Что нужно знать про ДП



Для того чтобы применить метод динамического программирования, задача должна характеризоваться двумя свойствами: иметь перекрывающиеся подзадачи и оптимальную структуру. Что это значит?

- Перекрывающиеся подзадачи: одни и те же задачи и подзадачи решаются многократно. Вместо того чтобы вычислять каждый раз заново, результаты сохраняют и используют повторно.

- Оптимальная подструктура: оптимальное решение задачи включает в себя оптимальные решения её подзадач. То есть зная оптимальные решения меньших частей задачи, можно построить оптимальное решение всей задачи.

Метод динамического программирования предоставляет ряд преимуществ — например, когда речь идёт о сложных задачах, где классические подходы неэффективны:

- Сокращение времени работы. Благодаря хранению результатов решения подзадач динамическое программирование экономит время на их повторное решение.

- Чёткая структура решения. Динамическое программирование предлагает чёткий план действий: определение состояния, переход между состояниями и вычисление результата.

Помимо этих преимуществ, метод обладает гибкостью и универсальностью. Он применим к широкому кругу задач — от оптимизации до структур данных и машинного обучения.

Ниже представлено несколько простых примеров задач, которые решаются с помощью динамического программирования.

### Пример 1. Кузнечик и лесенка

    Кузнечик стоит на 0-й ступеньке и может прыгать на одну или две ступеньки вперед.
    Сколько существует способов добраться до n-й ступеньки?

Логика решения этой задачи такая: на каждую ступеньку можно попасть либо с предыдущей, либо с "позапредыдущей", каждый раз только одним способом. Значит, количество способов попасть на n-ю ступеньку равно сумме количества способов попасть (n-1)-ю ступеньку и количества способов попасть (n-2)-ю ступеньку.

Ничего не напоминает?

Да это же ряд Фибоначчи!

Как мы уже знаем, вычислять числа Фибоначчи можно по-разному. Давайте посмотрим на самый плохой способ это делать - "наивную" рекурсию. И заодно замерим время для небольших чисел (я специально закодила принудительную остановку на больших числах, чтобы нам все не уронить).

In [1]:
from time import time

In [2]:
def ways_recur(n):
    if n > 40:
        return "Слишком много считать!"
    if n <= 1:
        return 1
    return ways_recur(n - 1) + ways_recur(n - 2)

nums = [10, 20, 30, 40]
for n in nums:
    t0 = time()
    print(f"для n = {n}: {ways_recur(n)} способов")
    print("время:", time() - t0)

для n = 10: 89 способов
время: 8.249282836914062e-05
для n = 20: 10946 способов
время: 0.0021283626556396484
для n = 30: 1346269 способов
время: 0.261751651763916
для n = 40: 165580141 способов
время: 34.95858335494995


Видите, как быстро растет время? Уже на 50 программа намертво встанет, даже не пробуйте.

Почему так? Потому что сложность рекурсивного алгоритма O(exp(n)).

Мы уже знаем один хороший способ делать это - через обычный цикл и две переменные, в которые последовательно присваиваем значения предыдущего и "позапредыдущего" числа (см. лекцию к теме 12).

Это действительно хороший алгоритм, но если мы хотим вычислить не просто одно значение, а целый ряд значений (как мы тут делаем), он все равно выполняет лишнюю работу. Значение для n=10 вычисляется снова, когда мы передаем n=20, потом то же самое для n=30, и т.д.

####Решение через "ДП сверху вниз" (мемоизация)

Можно поступить по-другому: сохранять в памяти уже вычисленные ранее значения и доставать их по мере надобности для вычисления следующих. Для этого достаточно просто завести словарь (кэш) и проверять его при входе, но есть встроенный более удобный инструмент для этого - декоратор lru_cache модуля functools.

Магия в том, что в этот декоратор можно обернуть ту самую рекурсивную функцию, и она станет работать мгновенно!

In [3]:
from functools import lru_cache

@lru_cache(maxsize=None)
# Параметр maxsize устанавливает максимальный размер кеша; если установлено None, кэш может расти без ограничений.
def ways_dynamic(n):
    # Проверка тут нам больше не нужна, мы легко вычислим значения для n>40
    if n <= 1:
        return 1
    return ways_dynamic(n - 1) + ways_dynamic(n - 2)

nums = [10, 20, 30, 40, 50]
for n in nums:
    t0 = time()
    print(f"для n = {n}: {ways_dynamic(n)} способов")
    print("время:", time() - t0)

для n = 10: 89 способов
время: 6.341934204101562e-05
для n = 20: 10946 способов
время: 1.7404556274414062e-05
для n = 30: 1346269 способов
время: 1.7404556274414062e-05
для n = 40: 165580141 способов
время: 1.5735626220703125e-05
для n = 50: 20365011074 способов
время: 1.0967254638671875e-05


Обратите внимание, что время почти не меняется с ростом чисел, т.е. каждый следующий шаг мы вычисляем столько же, сколько предыдущий - сложность O(n).

In [ ]:
nums = [60, 70, 80, 90, 100]
for n in nums:
    t0 = time()
    print(f"для n = {n}: {ways_dynamic(n)} способов")
    print("время:", time() - t0)

для n = 60: 2504730781961 способов
время: 7.271766662597656e-05
для n = 70: 308061521170129 способов
время: 1.71661376953125e-05
для n = 80: 37889062373143906 способов
время: 1.1920928955078125e-05
для n = 90: 4660046610375530309 способов
время: 1.430511474609375e-05
для n = 100: 573147844013817084101 способов
время: 1.0013580322265625e-05


### Пример 2. Размен монет.

    У вас есть монеты номиналом 1, 3 и 4 рубля. Нужно набрать сумму 6 рублей минимальным количеством монет.

Эта задача демонстрирует наглядный провал жадного алгоритма. Интуитивно хочется взять самую крупную монету 4, а затем добрать единицами: 4 + 1 + 1 (итого 3 монеты). Но правильный ответ: 3 + 3 (2 монеты). Это, конечно, легко посчитать в уме.

А что если на нужно набрать сумму в n монет?

####Решение через "ДП снизу вверх" (табуляция)

Рекурсия не всегда является лучшим способом решить задачу, даже в парадигме ДП. Дело в том, что рекурсия имеет аппаратное ограничение на глубину (мы об этом говорили), а нам может быть нужно посчитать значение для какого угодно большого числа. Поэтому мы здесь используем не рекурсию, а таблицу: будем последовательно находить минимальное число монет для каждой суммы от 1 до n, опираясь на уже посчитанные меньшие суммы.

In [ ]:
def min_coins(coins, target_amount):
  # Инициализируем таблицу состояний (массив dp) значением «бесконечность».
  # В индексах массива у нас будут суммы, а в значениях — минимальное число монет.
  dp = [float('inf')] * (target_amount + 1)

  # Базовый случай: для суммы 0 нужно 0 монет.
  dp[0] = 0

  # Начальная таблица
  print(f'Старт: {dp}\n')

  # Последовательно решаем задачу для каждой суммы от 1 до target_amount
  for current_sum in range(1, target_amount + 1):
    # Пробуем применить каждую доступную монету
    for coin in coins:
      # Проверяем, не превышает ли номинал новой монеты текущую сумму
      if current_sum >= coin:
        # Откатываем сумму назад на номинал новой монеты
        prev_sum = current_sum - coin
        # Сравниваем текущее лучшее решение с новым вариантом (откаченное состяние + 1 монета)
        dp[current_sum] = min(dp[current_sum], dp[prev_sum] + 1)

    # Для наглядности выводим каждый раз таблицу
    print(f'После расчета суммы {current_sum}: {dp}')

  # Если в целевой ячейке осталась бесконечность, размен невозможен
  if dp[target_amount] == float('inf'):
    return -1

  return dp[target_amount]

print(min_coins([1, 3, 4], 6))

Старт: [0, inf, inf, inf, inf, inf, inf]

После расчета суммы 1: [0, 1, inf, inf, inf, inf, inf]
После расчета суммы 2: [0, 1, 2, inf, inf, inf, inf]
После расчета суммы 3: [0, 1, 2, 1, inf, inf, inf]
После расчета суммы 4: [0, 1, 2, 1, 1, inf, inf]
После расчета суммы 5: [0, 1, 2, 1, 1, 2, inf]
После расчета суммы 6: [0, 1, 2, 1, 1, 2, 2]
2


Существует также другая версия этой задачи - поиск количества способов разменять сумму монетами фиксированного номинала.

Подробный разбор концепции и визуализацию можно изучить в статье про [Динамическое программирование на примере размена монет](https://codechick.io/journal/dinamicheskoe-programmirovanie-na-primere-razmena-monet).

###Пример 3. Ограбление домов.

    Вдоль улицы стоят дома, в каждом лежит известная сумма денег: [2, 7, 9, 3, 1].
    Сигнализация сработает, если ограбить два соседних дома.
    Какую максимальную сумму можно унести?

Ловушка этой задачи состоит в том, что нельзя просто сложить все четные или все нечетные дома (например, при суммах [2, 1, 1, 2] выгодно взять первый и последний дом: 2 + 2 = 4). Нельзя также просто взять самые богатые дома (в нашем примере они соседи). В простых случаях можно перебрать варианты, но что если у нас длинная улица?

####Решение в ДП с бинарным выбором.

Вор двигается по улице последовательно. Дойдя до дома i, он выбирает лучший из двух вариантов: не грабить этот дом (остаться с добычей, взятой на (i-1)-м шаге) или ограбить его, прибавив его деньги к добыче, собранной через один дом назад (на (i-2)-м шаге). Дойдя таким образом до конца улицы, он получит максимальную сумму.


In [ ]:
def rob_houses(houses):

  if not houses:
    return 0

  n = len(houses)

  # Если дом всего один, просто грабим его и возвращаем сумму
  if n == 1:
    return houses[0]

  # Инициализируем нулями массив, который будет хранить максимальную сумму,
  # которую можно унести, дойдя до дома с индексом i (включительно).
  dp = [0] * n

  # Базовые случаи для первых двух домов:
  dp[0] = houses[0]
  dp[1] = max(houses[0], houses[1])  # Выбираем более богатый из двух

  print(f'Старт: {dp}\n') # Для наглядности выводим таблицу

  # Последовательно принимаем решение для каждого дома, начиная с индекса 2
  for i in range(2, n):
    # Вариант А: грабим текущий дом i.
    # Тогда соседний дом (i - 1) трогать нельзя, берем добычу с шага (i - 2).
    rob_current = houses[i] + dp[i - 2]

    # Вариант Б: пропускаем текущий дом i.
    # Тогда сохраняем лучшую добычу, которая была на предыдущем шаге (i - 1).
    skip_current = dp[i - 1]

    # Записываем в индекс i лучший из двух вариантов
    dp[i] = max(rob_current, skip_current)

    print(f'Дом {i} (денег: {houses[i]}): грабим ({rob_current}) vs пропускаем'
        f' ({skip_current}) -> {dp}')

  # Итоговый ответ для всей улицы лежит в самой последней ячейке
  return dp[-1]

print(rob_houses([2, 7, 9, 3, 1]))

Старт: [2, 7, 0, 0, 0]

Дом 2 (денег: 9): грабим (11) vs пропускаем (7) -> [2, 7, 11, 0, 0]
Дом 3 (денег: 3): грабим (10) vs пропускаем (11) -> [2, 7, 11, 11, 0]
Дом 4 (денег: 1): грабим (12) vs пропускаем (11) -> [2, 7, 11, 11, 12]
12


Обратите внимание, что для расчета dp[i] нужны только две предыдущие ячейки (dp[i - 1] и dp[i - 2]). Это значит, что целый массив использовать не обязательно — задачу можно сжать до двух переменных и сократить память с \(O(n)\) до \(O(1)\).

###Причем тут лингвистика? Расстояние Левенштейна.

Пусть через канал связи передается какой-либо сигнал, состоящий из последовательности символов.
При передаче сигнала возможны искажения - некоторые символы заменяются на другие.

Мера «близости» двух строк, равная числу различий между ними, то есть количеству соответствующих символов в последовательностях, которые различаются, называется **расстоянием Хемминга**.

Если разрешать не только менять символы, но также удалять и вставлять символы, то минимальное количество изменений (замена символа, удаление символа, вставка символа), которое необходимо сделать с одной строкой, чтобы получить другую строку, называется **расстоянием Левенштейна** (в честь российского математика Владимира Левенштейна) или **расстоянием редактирования**.

Расстояние Левенштейна появляется, например, в системах автоматической проверки орфографии.

Если при наборе текста человек допустил одну или несколько ошибок, то набранное им слово будет отсутствовать в словаре. При этом нужно понять, какое слово наборщик имел в виду.

Если считать, что наиболее типичные ошибки при наборе текста - это пропуск символа, вставка лишнего символа или замена одного символа на другой, то необходимо неправильное слово заменить на слово из словаря с минимальным расстоянием Левенштейна.

Так вот: *расстояние Левенштейна ищется динамически!*

В чистом Python нет готовых функций для поиска расстояния Левенштейна, но существуют сторонние библиотеки, где это есть. В качестве примера приведу классическую библиотеку Levenshtein. Чтобы ее установить, пишем:

    pip install Levenshtein

Если работаете в Колабе, то команду для установки библиотеки надо ввести в отдельной кодовой ячейке, и перед ней поставить восклицательный знак:

In [ ]:
!pip install Levenshtein

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 43.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [Levenshtein]


Эта библиотека написана на С именно в парадигме динамического программирования, поэтому она работает очень быстро.

In [ ]:
import Levenshtein

dist = Levenshtein.distance("кот", "скат")
t0 = time()
print(dist)
print(time() - t0)

2
0.00016021728515625


Но использовать готовые решения - это чит. Сегодня мы с вами научимся сами писать функцию для поиска расстояния Левенштейна в ДП.

### Алгоритм поиска расстояния Левенштейна

#### Допустимые операции
Для преобразования строки $A$ длины $m$ в строку $B$ длины $n$ разрешены три базовые операции:
1. **Вставка** одного символа (Insertion).
2. **Удаление** одного символа (Deletion).
3. **Замена** одного символа на другой (Substitution).

---

#### Математическая постановка задачи

Пусть даны две строки:
* $A = a_1 a_2 \dots a_m$ (длина $m$)
* $B = b_1 b_2 \dots b_n$ (длина $n$)

Введем двумерную матрицу $D$ размером $(m + 1) \times (n + 1)$, где элемент $D(i, j)$ означает **расстояние Левенштейна между префиксом строки $A$ длины $i$ и префиксом строки $B$ длины $j$**.

1. Базовые случаи (края матрицы)
Превращение непустой строки в пустую (и наоборот) требует столько операций, какова длина непустой строки:

$$D(0, 0) = 0$$

$$D(i, 0) = i, \quad \text{для всех } 1 \le i \le m \quad \text{(удаление } i \text{ символов из } A\text{)}$$

$$D(0, j) = j, \quad \text{для всех } 1 \le j \le n \quad \text{(вставка } j \text{ символов из } B\text{)}$$

2. Рекуррентная формула перехода
Для всех $1 \le i \le m$ и $1 \le j \le n$ значение ячейки $D(i, j)$ вычисляется на основе трех уже известных соседних ячеек:

$$D(i, j) = \min \begin{cases}
D(i - 1, j) + 1 & \text{(удаление символа } a_i\text{)} \\
D(i, j - 1) + 1 & \text{(вставка символа } b_j\text{)} \\
D(i - 1, j - 1) + \text{cost}(a_i, b_j) & \text{(совпадение или замена)}
\end{cases}$$

где функция стоимости замены $\text{cost}(a_i, b_j)$ равна $0$, если символы совпадают, и $1$, если они различны:

$$\text{cost}(a_i, b_j) = \begin{cases}
0, & \text{если } a_i = b_j \\
1, & \text{если } a_i \neq b_j
\end{cases}$$

3. Финальный ответ
Итоговое расстояние между полными строками $A$ и $B$ находится в правом нижнем углу матрицы:

$$\text{Levenshtein}(A, B) = D(m, n)$$

---

#### Пример пошагового разбора: превращаем «КОТ» в «СКАТ»

* Строка $A = \text{«КОТ»}$ ($m = 3$)
* Строка $B = \text{«СКАТ»}$ ($n = 4$)

Заполним таблицу $D$ размером $4 \times 5$ (символ $\varepsilon$ обозначает пустую строку):

| | $\varepsilon$ | **С** | **К** | **А** | **Т** |
| :---: | :---: | :---: | :---: | :---: | :---: |
| **$\varepsilon$** | **0** | 1 | 2 | 3 | 4 |
| **К** | 1 | **1** | **1** | 2 | 3 |
| **О** | 2 | 2 | 2 | **2** | 3 |
| **Т** | 3 | 3 | 3 | 3 | **2** |

**Как получены ключевые значения:**
1. **$D(1, 1)$ (превращаем «К» в «С»):** буквы не равны ($\text{cost} = 1$). Берем минимум из трех вариантов:
   * Удаление: $D(0, 1) + 1 = 1 + 1 = 2$
   * Вставка: $D(1, 0) + 1 = 1 + 1 = 2$
   * Замена: $D(0, 0) + 1 = 0 + 1 = 1$
   * Итог: $D(1, 1) = 1$.
2. **$D(1, 2)$ (превращаем «К» в «СК»):** последние буквы совпадают ($\text{«К»} = \text{«К»}$, $\text{cost} = 0$). Диагональный переход дает $D(0, 1) + 0 = 1$.
3. **$D(2, 3)$ (превращаем «КО» в «СКА»):** заменяем «О» на «А» ($\text{cost} = 1$). Диагональный переход от $D(1, 2)$ дает $1 + 1 = 2$.
4. **$D(3, 4)$ (превращаем «КОТ» в «СКАТ»):** буквы «Т» совпадают ($\text{cost} = 0$). Диагональный переход от $D(2, 3)$ дает $2 + 0 = 2$.

**Ответ:** $D(3, 4) = 2$ операции (1. Вставить **«С»** в начало $\rightarrow$ *«СКОТ»*; 2. Заменить **«О»** на **«А»** $\rightarrow$ *«СКАТ»*).


На основании данного алгоритма попробуем заполнить пропуски в коде ниже.

In [6]:
def levenshtein_distance(a: str, b: str) -> int:
    m, n = len(a), len(b)

    # Создаем таблицу D размером (m + 1) x (n + 1), заполненную нулями
    dp = [[0] * (n + 1) for _ in range(m + 1)]

    # Заполняем базовые случаи (первый столбец и первую строку)
    for i in range(m + 1):
        dp[i][0] = i  # ЗАДАНИЕ 1: сколько операций нужно, чтобы из префикса длины i получить пустую строку?

    for j in range(n + 1):
        dp[0][j] = j  # ЗАДАНИЕ 2: сколько операций нужно, чтобы из пустой строки получить префикс длины j?

    # Заполняем основную часть таблицы
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            # Считаем стоимость замены (cost)
            # ВНИМАНИЕ: в таблице индексы с 1, а в строках Python — с 0!
            if a[i - 1] == b[j - 1]:
                cost = 0  # ЗАДАНИЕ 3: символы совпали
            else:
                cost = 1  # ЗАДАНИЕ 4: символы разные

            # Рекуррентная формула перехода (минимум из трех соседей)
            dp[i][j] = min(
                dp[i-1][j] + 1,  # Удаление
                dp[i][j-1] + 1,  # Вставка
                dp[i-1][j-1] + cost   # Замена или совпадение (шаг по диагонали)
            )  # ЗАДАНИЕ 5

    for i in range(m+1):
      for j in range(n+1):
        print(dp[i][j], end=" ")
      print()

    # Возвращаем финальный ответ из правого нижнего угла
    return dp[m][n]

print("Расстояние между КОРОВА и ЗДОРОВО:", levenshtein_distance("КОРОВА", "ЗДОРОВО"))
#должно выдать 2


0 1 2 3 4 5 6 7 
1 1 2 3 4 5 6 7 
2 2 2 2 3 4 5 6 
3 3 3 3 2 3 4 5 
4 4 4 3 3 2 3 4 
5 5 5 4 4 3 2 3 
6 6 6 5 5 4 3 3 
Расстояние между КОРОВА и ЗДОРОВО: 3


Ну как, несложно?

А теперь попробуйте посчитать класс алгоритма, который мы только что записали, по быстродействию и по памяти. Можно ли его как-то оптимизировать?

#### Дополнение: перестановки. Дамерау-Левенштейн.

Какое расстояние Левенштейна выдаст наш алгоритм для слов „КОТ“ и „КТО“?

А сколько опечаток реально сделал человек при печати?

Чтобы учитывать такие вещи, используется модифицированная метрика, **расстояние Дамерау-Левенштейна**, где к трем базовым операциям добавляется четвертая — перестановка двух соседних символов.

**Вопрос на подумать:** Как нужно изменить наш алгоритм, чтобы он считал Дамерау-Левенштейна вместо обычного Левенштейна?
